# External benchmark: MAVOS-DD test sample, by generator, language and compression tier

Scores the released system on a stratified sample of the MAVOS-DD test split (7+ generators, 8 languages, YouTube reals), re-encoded at H.264 CRF 23 and 40 as extra tiers, and reports by generator, language and source video with video-clustered intervals. Accept the dataset terms at https://huggingface.co/datasets/unibuc-cs/MAVOS-DD with your account first; the login cell asks for a token.

In [ ]:
!nvidia-smi -L; nproc
%cd /content
!rm -rf repo && git clone -q -b revision/round-3 https://github.com/saoirsebarry/multiagent-deepfake-detection.git repo
%cd /content/repo
!pip -q install speechbrain timm librosa opencv-python-headless mtcnn datasets huggingface_hub
!apt-get -qq install -y ffmpeg > /dev/null
!git log --oneline -1
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
from huggingface_hub import login
login()  # paste a read token from https://huggingface.co/settings/tokens

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/polyglotfake/external/mavos'
os.makedirs(OUT, exist_ok=True)

Only the metadata files are snapshotted (a few MB); the selected videos are downloaded one by one into Drive, so a recycled runtime resumes where it stopped instead of starting over. The download needs no GPU; switch the runtime to a T4 before the staging cell.


In [ ]:
from huggingface_hub import snapshot_download
SNAP = snapshot_download('unibuc-cs/MAVOS-DD', repo_type='dataset', local_dir='/content/mavos_meta', allow_patterns=['*.json', '*.arrow', '*.py', 'README.md'])
!python -u tools/external_benchmark/mavos_manifest.py --snapshot /content/mavos_meta --out "$OUT/videos.csv" --cap 40 --real_cap 40 --video_dir "$OUT/videos"


## Stage clips (original, CRF 23, CRF 40)

## Stage and score one tier at a time
Each tier is staged on local disk and scored immediately, so a recycled runtime loses at most one tier; a tier whose `scores.csv` is already on Drive is skipped. The original tier stages the videos as downloaded; the CRF tiers re-encode them with x264 first (the FaceForensics++ convention).

In [ ]:
import os
TIER, CRF = 'original', None
N = max(2, os.cpu_count() // 2); os.environ['N'] = str(N); os.environ['TIER'] = TIER; os.environ['CRFARG'] = '' if CRF is None else f'--crf {CRF}'
import subprocess, glob
if not os.path.exists(f'{OUT}/{TIER}/scores.csv'):
    !seq 0 $((N-1)) | xargs -P $N -I{} sh -c 'python -u tools/external_benchmark/stage_videos.py --manifest "$OUT/videos.csv" --out_dir /content/mavos_clips $CRFARG --shard {}/$N > /content/stage_${TIER}_{}.log 2>&1'
    !cp /content/mavos_clips/metadata.csv "$OUT/"; mkdir -p "$OUT/$TIER"
    D = '/content/mavos_clips' if CRF is None else f'/content/mavos_clips/crf{CRF}'
    os.environ['D'] = D
    !ls $D/test | wc -l
    !python -u tools/source_disjoint/score_split.py --data_dir $D --split test --ckpt_dir checkpoints --out "$OUT/$TIER/scores.csv" 2>&1 | tail -n 5
else:
    print(TIER, 'already scored')
!wc -l "$OUT/$TIER/scores.csv"


In [ ]:
import os
TIER, CRF = 'crf23', 23
N = max(2, os.cpu_count() // 2); os.environ['N'] = str(N); os.environ['TIER'] = TIER; os.environ['CRFARG'] = '' if CRF is None else f'--crf {CRF}'
import subprocess, glob
if not os.path.exists(f'{OUT}/{TIER}/scores.csv'):
    !seq 0 $((N-1)) | xargs -P $N -I{} sh -c 'python -u tools/external_benchmark/stage_videos.py --manifest "$OUT/videos.csv" --out_dir /content/mavos_clips $CRFARG --shard {}/$N > /content/stage_${TIER}_{}.log 2>&1'
    !cp /content/mavos_clips/metadata.csv "$OUT/"; mkdir -p "$OUT/$TIER"
    D = '/content/mavos_clips' if CRF is None else f'/content/mavos_clips/crf{CRF}'
    os.environ['D'] = D
    !ls $D/test | wc -l
    !python -u tools/source_disjoint/score_split.py --data_dir $D --split test --ckpt_dir checkpoints --out "$OUT/$TIER/scores.csv" 2>&1 | tail -n 5
else:
    print(TIER, 'already scored')
!wc -l "$OUT/$TIER/scores.csv"


In [ ]:
import os
TIER, CRF = 'crf40', 40
N = max(2, os.cpu_count() // 2); os.environ['N'] = str(N); os.environ['TIER'] = TIER; os.environ['CRFARG'] = '' if CRF is None else f'--crf {CRF}'
import subprocess, glob
if not os.path.exists(f'{OUT}/{TIER}/scores.csv'):
    !seq 0 $((N-1)) | xargs -P $N -I{} sh -c 'python -u tools/external_benchmark/stage_videos.py --manifest "$OUT/videos.csv" --out_dir /content/mavos_clips $CRFARG --shard {}/$N > /content/stage_${TIER}_{}.log 2>&1'
    !cp /content/mavos_clips/metadata.csv "$OUT/"; mkdir -p "$OUT/$TIER"
    D = '/content/mavos_clips' if CRF is None else f'/content/mavos_clips/crf{CRF}'
    os.environ['D'] = D
    !ls $D/test | wc -l
    !python -u tools/source_disjoint/score_split.py --data_dir $D --split test --ckpt_dir checkpoints --out "$OUT/$TIER/scores.csv" 2>&1 | tail -n 5
else:
    print(TIER, 'already scored')
!wc -l "$OUT/$TIER/scores.csv"


In [ ]:
!python -u tools/external_benchmark/report_by_group.py --scores "$OUT/original/scores.csv" "$OUT/crf23/scores.csv" "$OUT/crf40/scores.csv" --metadata "$OUT/metadata.csv" --group_by generator language source_video --cluster source_video --tau 0.35 --out "$OUT/report.json"